# 🎸 MOD Universal Plugin Porter & Cloud Cross-Compiler
### Automated Multi-Architecture LV2 Re-Packager for MOD Desktop (Windows/Linux/macOS) & MODEP (Raspberry Pi / Patchbox OS)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danny1marshall1587-maker/mod-universal-plugin-hub/blob/main/notebooks/MOD_Universal_Porter.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE)
[![MOD Compatible](https://img.shields.io/badge/MOD-Desktop%20%26%20MODEP%20Ready-00ff66.svg)]()

---
### 🌟 What this Notebook does:
1. **Single Plugin Mode**: Pastes any LV2 GitHub repository or ZIP URL to cross-compile and download.
2. **Batch Transpiler Mode**: 1-click batch conversion of the **Top 38 MODEP / Blokas ARM Plugins** into Universal FAT bundles.
3. **Cross-compiles in parallel** for all MOD platforms:
   - **🪟 Windows 64-bit** (`.dll` for MOD Desktop on Windows)
   - **🐧 Linux x86_64** (`.so` for MOD Desktop on Linux)
   - **🍓 Raspberry Pi 3/4 (ARMv7 32-bit)** (`.so` for Blokas MODEP / Patchbox OS)
   - **⚡ Raspberry Pi 4/5 (AArch64 64-bit)** (`.so` for 64-bit MODEP / MOD Dwarf / Duo X)
4. **Auto-Synthesizes MODGUI Pedal Layouts**: Generates touch-ready HTML5/CSS3 pedal graphics with rotating knobs and footswitches.
5. **1-Click Download**: Automatically downloads the finished Universal FAT `.zip` bundles straight to your PC!

## ⚙️ Step 1: Install Cross-Compilers & Audio DSP Dependencies
Run this cell once to set up the Ubuntu cloud build environment (takes ~25 seconds).

In [ ]:
# @title 📦 Setup Cross-Compilation Toolchains
import os, sys, subprocess

print("[*] Updating package lists and installing cross-compilers...")
!apt-get update -qq
!apt-get install -y -qq \
    build-essential \
    gcc g++ \
    gcc-arm-linux-gnueabihf g++-arm-linux-gnueabihf \
    gcc-aarch64-linux-gnu g++-aarch64-linux-gnu \
    gcc-mingw-w64 g++-mingw-w64 \
    cmake make git curl zip tar jq \
    lv2-dev libfftw3-dev libsndfile1-dev \
    faust > /dev/null

# Download sse2neon header for seamless x86 SSE vector translation on ARM
os.makedirs("/usr/local/include/sse2neon", exist_ok=True)
!curl -sL https://raw.githubusercontent.com/DLTcollab/sse2neon/master/sse2neon.h -o /usr/local/include/sse2neon/sse2neon.h
!cp /usr/local/include/sse2neon/sse2neon.h /usr/local/include/sse2neon.h

# Clone or pull latest hub scripts
!git clone --depth 1 https://github.com/danny1marshall1587-maker/mod-universal-plugin-hub.git /content/mod-hub-repo -q 2>/dev/null || (cd /content/mod-hub-repo && git pull -q)

print("\n[+] Environment Ready! All 4 target toolchains installed successfully.")

## 🎛️ Step 2: Single Plugin Porter
Paste any single GitHub repository or ZIP URL to compile and package on demand.

In [ ]:
# @title 🚀 Single Plugin Multi-Architecture Porter
PLUGIN_SOURCE_URL = "https://github.com/danny1marshall1587-maker/cyber-blues-driver-lv2" # @param {type:"string"}
CUSTOM_PLUGIN_NAME = "" # @param {type:"string"}
PEDAL_COLOR_THEME = "copper" # @param ["copper", "blue", "gold", "green", "orange", "black", "silver", "purple", "red"]
AUTO_DOWNLOAD_ZIP = True # @param {type:"boolean"}

!cd /content/mod-hub-repo && git pull -q 2>/dev/null

!python3 /content/mod-hub-repo/scripts/porter_engine.py \
    --source "{PLUGIN_SOURCE_URL}" \
    --name "{CUSTOM_PLUGIN_NAME}" \
    --theme "{PEDAL_COLOR_THEME}" \
    --output-dir "/content/single_output"

if AUTO_DOWNLOAD_ZIP:
    from google.colab import files
    import glob
    zips = glob.glob("/content/single_output/*.zip")
    if zips:
        print("\n[⬇] Starting 1-Click Browser Download...")
        files.download(zips[0])

## 🏭 Step 3: Batch Transpile Top 38 MODEP Plugins in 1 Click
Compile and package the entire catalog of top 38 Blokas/MODEP plugins into a single Universal FAT Mega-Bundle.

In [ ]:
# @title ⚡ 1-Click Batch Transpile Entire MODEP Library
BATCH_LIMIT = 0 # @param {type:"integer"} 
# @markdown *(Set `0` for ALL 38 plugins, or e.g. `5` for quick testing)*

import os, zipfile
from google.colab import files

!cd /content/mod-hub-repo && git pull -q 2>/dev/null

!python3 /content/mod-hub-repo/scripts/modep_batch_porter.py \
    --catalog /content/mod-hub-repo/scripts/modep_catalog.json \
    --limit {BATCH_LIMIT} \
    --output-dir /content/batch_universal_plugins

# Create all-in-one Mega-Bundle ZIP
mega_zip = "/content/MODEP-Top-38-Universal-FAT-MegaBundle.zip"
print("\n[*] Compressing all ported plugins into Mega-Bundle ZIP...")
with zipfile.ZipFile(mega_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_list in os.walk("/content/batch_universal_plugins"):
        for f in files_list:
            if f.endswith('.zip') or f.endswith('.ttl') or f.endswith('.so') or f.endswith('.dll'):
                fpath = os.path.join(root, f)
                rpath = os.path.relpath(fpath, "/content/batch_universal_plugins")
                zf.write(fpath, rpath)

print(f"\n[+] Mega-Bundle Created: {mega_zip} ({os.path.getsize(mega_zip):,} bytes)")
print("[⬇] Starting 1-Click Download to your computer...")
files.download(mega_zip)